# **1 Sample construction**

In [3]:
# Importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

In [4]:
# Define the path to the data file (relative path)
data_path = Path("public.dat")

# Column names from the codebook and data preparation
columns = [
    "SHEET", "CHAIN", "CO_OWNED", "STATE",
    "SOUTHJ", "CENTRALJ", "NORTHJ", "PA1", "PA2", "SHORE",
    "NCALLS", "EMPFT", "EMPPT", "NMGRS", "WAGE_ST",
    "INCTIME", "FIRSTINC", "BONUS", "PCTAFF", "MEALS",
    "OPEN", "HRSOPEN", "PSODA", "PFRY", "PENTREE",
    "NREGS", "NREGS11",
    "TYPE2", "STATUS2", "DATE2", "NCALLS2",
    "EMPFT2", "EMPPT2", "NMGRS2", "WAGE_ST2",
    "INCTIME2", "FIRSTIN2", "SPECIAL2", "MEALS2",
    "OPEN2R", "HRSOPEN2", "PSODA2", "PFRY2",
    "PENTREE2", "NREGS2", "NREGS112"
]

# Positions given in the codebook
colspecs = [
    (0, 3), (4, 5), (6, 7), (8, 9),
    (10, 11), (12, 13), (14, 15), (16, 17),
    (18, 19), (20, 21),
    (22, 24), (25, 30), (31, 36), (37, 42),
    (43, 48), (49, 54), (55, 60), (61, 62),
    (63, 68), (69, 70), (71, 76), (77, 82),
    (83, 88), (89, 94), (95, 100), (101, 103),
    (104, 106), (107, 108), (109, 110), (111, 117),
    (118, 120), (121, 126), (127, 132), (133, 138),
    (139, 144), (145, 150), (151, 156), (157, 158),
    (159, 160), (161, 166), (167, 172), (173, 178),
    (179, 184), (185, 190), (191, 193), (194, 196)
]

df = pd.read_fwf(
    data_path,
    colspecs=colspecs,
    names=columns,
    header=None
)

# Replace "." by NaN
df = df.replace(".", np.nan)

df.to_csv("public_clean.csv", index=False)

print(df.head(10))

   SHEET  CHAIN  CO_OWNED  STATE  SOUTHJ  CENTRALJ  NORTHJ  PA1  PA2  SHORE  \
0     46      1         0      0       0         0       0    1    0      0   
1     49      2         0      0       0         0       0    1    0      0   
2    506      2         1      0       0         0       0    1    0      0   
3     56      4         1      0       0         0       0    1    0      0   
4     61      4         1      0       0         0       0    1    0      0   
5     62      4         1      0       0         0       0    1    0      0   
6    445      1         0      0       0         0       0    0    1      0   
7    451      1         0      0       0         0       0    0    1      0   
8    455      2         1      0       0         0       0    0    1      0   
9    458      2         1      0       0         0       0    1    0      0   

   ...  FIRSTIN2 SPECIAL2 MEALS2 OPEN2R HRSOPEN2 PSODA2 PFRY2  PENTREE2  \
0  ...      0.08        1      2   6.50    16.50   1.03

In [5]:
# Number of restaurants in each state

numb_restaurants_nj = df[df["STATE"] == 1].shape[0]
print(f"The number of restaurants in New Jersey is {numb_restaurants_nj}")

numb_restaurants_pa = df[df["STATE"] == 0].shape[0]
print(f"The number of restaurants in Pennsylvania is {numb_restaurants_pa}")

The number of restaurants in New Jersey is 331
The number of restaurants in Pennsylvania is 79


# **3 Starting wages**

In [6]:
# We convert the wages to numerical values
df['WAGE_ST'] = pd.to_numeric(df['WAGE_ST'], errors='coerce')
df['WAGE_ST2'] = pd.to_numeric(df['WAGE_ST2'], errors='coerce')


df_grouped = df.groupby(by ='STATE')[['WAGE_ST', 'WAGE_ST2']].mean()

print('Average starting wage in each state before and after the policy\n\n' , df_grouped)

Average starting wage in each state before and after the policy

         WAGE_ST  WAGE_ST2
STATE                    
0      4.630132  4.617465
1      4.612134  5.080849


In [7]:
print('Number of number of observations used in the result of NJ before.\n', df[df["STATE"] == 1]["WAGE_ST"].dropna().shape[0] )

print('Number of number of observations used in the result of NJ after.\n', df[df["STATE"] == 1]["WAGE_ST2"].dropna().shape[0] )

print('Number of number of observations used in the result of PA before.\n', df[df["STATE"] == 0]["WAGE_ST"].dropna().shape[0] )

print('Number of number of observations used in the result of PA after.\n', df[df["STATE"] == 0]["WAGE_ST2"].dropna().shape[0] )

Number of number of observations used in the result of NJ before.
 314
Number of number of observations used in the result of NJ after.
 318
Number of number of observations used in the result of PA before.
 76
Number of number of observations used in the result of PA after.
 71


# **4 Difference in differences**

In [8]:
# We convert the necessary columns to numerical values
df['EMPFT'] = pd.to_numeric(df['EMPFT'], errors='coerce')
df['EMPFT2'] = pd.to_numeric(df['EMPFT2'], errors='coerce')

df['EMPPT'] = pd.to_numeric(df['EMPPT'], errors='coerce')
df['EMPPT2'] = pd.to_numeric(df['EMPPT2'], errors='coerce')

df['NMGRS'] = pd.to_numeric(df['NMGRS'], errors='coerce')
df['NMGRS2'] = pd.to_numeric(df['NMGRS2'], errors='coerce')

In [9]:
# We create two columns for the FTE before and after the policy

df['FTE'] = df['EMPFT'] + 0.5*df['EMPPT'] + df['NMGRS']
df['FTE2'] = df['EMPFT2'] + 0.5*df['EMPPT2'] + df['NMGRS2']

df.head(5)

,SHEET,CHAIN,CO_OWNED,STATE,SOUTHJ,CENTRALJ,NORTHJ,PA1,PA2,SHORE,...,MEALS2,OPEN2R,HRSOPEN2,PSODA2,PFRY2,PENTREE2,NREGS2,NREGS112,FTE,FTE2
0,46,1,0,0,0,0,0,1,0,0,...,2,6.50,16.50,1.03,NaN,0.94,4,4,40.50,24.0
1,49,2,0,0,0,0,0,1,0,0,...,2,10.00,13.00,1.01,0.89,2.35,4,4,13.75,11.5
2,506,2,1,0,0,0,0,1,0,0,...,1,11.00,11.00,0.95,0.74,2.33,4,3,8.50,10.5
3,56,4,1,0,0,0,0,1,0,0,...,2,10.00,12.00,0.92,0.79,0.87,2,2,34.00,20.0
4,61,4,1,0,0,0,0,1,0,0,...,2,10.00,12.00,1.01,0.84,0.95,2,2,24.00,35.5


In [10]:
df_grouped = df.groupby(by ='STATE')[['FTE', 'FTE2']].mean()
print('FTE employment in each state before and after the policy\n\n', df_grouped)

FTE employment in each state before and after the policy

              FTE       FTE2
STATE                      
0      23.331169  21.165584
1      20.439408  21.027429


In [11]:
print('Change in Pennsylvania : ', df_grouped['FTE2'][0] - df_grouped['FTE'][0])
print('Change in New Jersey : ', df_grouped['FTE2'][1] - df_grouped['FTE'][1])

Change in Pennsylvania :  -2.1655844155844157
Change in New Jersey :  0.5880213673961663


In [12]:
DiD = (df_grouped['FTE2'][1] - df_grouped['FTE'][1]) - (df_grouped['FTE2'][0] - df_grouped['FTE'][0])
print('Difference-in-differences:', DiD)

Difference-in-differences: 2.753605782980582


In [13]:
print('Number of number of observations used in the result of NJ before.\n', df[df["STATE"] == 1]["FTE"].dropna().shape[0] )

print('Number of number of observations used in the result of NJ after.\n', df[df["STATE"] == 1]["FTE2"].dropna().shape[0] )

print('Number of number of observations used in the result of PA before.\n', df[df["STATE"] == 0]["FTE"].dropna().shape[0] )

print('Number of number of observations used in the result of PA after.\n', df[df["STATE"] == 0]["FTE2"].dropna().shape[0] )

Number of number of observations used in the result of NJ before.
 321
Number of number of observations used in the result of NJ after.
 319
Number of number of observations used in the result of PA before.
 77
Number of number of observations used in the result of PA after.
 77


# **5 Linear Regression**

In [14]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Build everything from df (no dependency on other cells)
d = df.copy()
for c in ["EMPFT", "EMPPT", "NMGRS", "EMPFT2", "EMPPT2", "NMGRS2"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["FTE"]  = d["EMPFT"]  + 0.5 * d["EMPPT"]  + d["NMGRS"]
d["FTE2"] = d["EMPFT2"] + 0.5 * d["EMPPT2"] + d["NMGRS2"]

# Balanced sample: FTE observed in both waves
bal = d.dropna(subset=["FTE", "FTE2"]).copy()
bal["Change_FTE"] = bal["FTE2"] - bal["FTE"]
bal["NJ"] = bal["STATE"].astype(int)
print("Balanced sample:", len(bal), "| NJ:", (bal.NJ == 1).sum(), "| PA:", (bal.NJ == 0).sum())

# First-difference regression
fd = smf.ols("Change_FTE ~ NJ", data=bal).fit()
print(fd.summary().tables[1])
print("N =", int(fd.nobs))

chg = bal.groupby("NJ")["Change_FTE"].mean()
print("Mean change PA:", round(chg[0], 3), "| NJ:", round(chg[1], 3), "| DiD:", round(chg[1] - chg[0], 3))

Balanced sample: 384 | NJ: 309 | PA: 75
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.2833      1.036     -2.205      0.028      -4.319      -0.247
NJ             2.7500      1.154      2.382      0.018       0.480       5.020
N = 384
Mean change PA: -2.283 | NJ: 0.467 | DiD: 2.75


In [15]:
before = bal[["SHEET", "NJ", "FTE"]].rename(columns={"FTE": "FTE_t"});  before["After"] = 0
after  = bal[["SHEET", "NJ", "FTE2"]].rename(columns={"FTE2": "FTE_t"}); after["After"] = 1
long = pd.concat([before, after], ignore_index=True)
long["NJ_After"] = long["NJ"] * long["After"]

stacked = smf.ols("FTE_t ~ NJ + After + NJ_After", data=long).fit(
    cov_type="cluster", cov_kwds={"groups": long["SHEET"]})
print(stacked.summary().tables[1])
print("N =", int(stacked.nobs))

print("b3 stacked:", round(stacked.params["NJ_After"], 6), "| b1 first-diff:", round(fd.params["NJ"], 6))
assert np.isclose(stacked.params["NJ_After"], fd.params["NJ"])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3800      1.382     16.917      0.000      20.671      26.089
NJ            -2.9494      1.475     -1.999      0.046      -5.841      -0.058
After         -2.2833      1.249     -1.828      0.068      -4.731       0.165
NJ_After       2.7500      1.338      2.055      0.040       0.127       5.373
N = 768
b3 stacked: 2.75 | b1 first-diff: 2.75


Interpretation : The intercept (−2.28) is the average change in Pennsylvania, the control group. The NJ coefficient (+2.75) is the difference between the average change in NJ and in PA, which is the difference-in-differences estimate: NJ restaurants gained about 2.75 more employees than the PA trend suggests. This effect is significant at 5% (p = 0.018), and it is the opposite of the standard prediction that a higher minimum wage lowers employment. The stacked regression gives the same estimate (NJ × After = 2.75), because it is the same comparison written differently. Its standard error is clustered by restaurant, since each restaurant appears twice.

# **6 Statistical Framework**

Hypotheses
H₀: β₁ = 0. The minimum-wage increase had no effect on employment: the change in FTE in NJ equals the change in PA.

H₁: β₁ ≠ 0. Two-sided test.
Economic theory predicts a negative effect, so you can also mention a one-sided alternative (H₁: β₁ < 0). The homework just says "test the null", so keep the two-sided test as the main one.

In [16]:
# H0 : beta1 = 0 (no effect of the minimum wage on employment)
# H1 : beta1 != 0

# Manual computation (to show the formulas), checked against statsmodels
b1 = fd.params['NJ']
se = fd.bse['NJ']
t_stat = b1 / se
dof = fd.df_resid
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), dof))
t_crit = stats.t.ppf(0.975, dof)
ci_low, ci_high = b1 - t_crit * se, b1 + t_crit * se

# Cross-check with statsmodels
assert np.isclose(t_stat, fd.tvalues['NJ'])
assert np.isclose(p_value, fd.pvalues['NJ'])
assert np.allclose([ci_low, ci_high], fd.conf_int().loc['NJ'].values)

# Same test with heteroskedasticity-robust standard errors (HC1)
model_rob = smf.ols('Change_FTE ~ NJ', data=bal).fit(cov_type='HC1')

# One-sided p-value for H1 : beta1 < 0
p_one_sided = stats.t.cdf(t_stat, dof)

# Summary table
results = pd.DataFrame({
    'Estimate': [b1, b1],
    'Std. error': [se, model_rob.bse['NJ']],
    't-stat': [t_stat, model_rob.tvalues['NJ']],
    'p-value': [p_value, model_rob.pvalues['NJ']],
    'CI 95% low': [ci_low, model_rob.conf_int().loc['NJ', 0]],
    'CI 95% high': [ci_high, model_rob.conf_int().loc['NJ', 1]],
    'N': [int(fd.nobs)] * 2
}, index=['OLS (classical SE)', 'OLS (robust HC1 SE)']).round(4)

print(results)
print(f'One-sided p-value (H1: beta1 < 0): {p_one_sided:.4f}')
print(f'Critical value (5%, two-sided): {t_crit:.3f} | degrees of freedom: {int(dof)}')

# Sanity check : Welch t-test on the change in FTE (NJ vs PA)
welch = stats.ttest_ind(bal.loc[bal.NJ == 1, 'Change_FTE'],
                        bal.loc[bal.NJ == 0, 'Change_FTE'],
                        equal_var=False)
print('Welch t-test :', welch)

                     Estimate  Std. error  t-stat  p-value  CI 95% low  \
OLS (classical SE)       2.75      1.1544  2.3823   0.0177      0.4803   
OLS (robust HC1 SE)      2.75      1.3377  2.0557   0.0398      0.1281   

                     CI 95% high    N  
OLS (classical SE)        5.0197  384  
OLS (robust HC1 SE)       5.3719  384  
One-sided p-value (H1: beta1 < 0): 0.9912
Critical value (5%, two-sided): 1.966 | degrees of freedom: 382
Welch t-test : TtestResult(statistic=np.float64(2.048659802022799), pvalue=np.float64(0.04319974156630396), df=np.float64(96.88377548977087))


# **7 Robustness**

In [17]:
# Same balanced sample as the baseline regression
robust_sample = bal.dropna(
    subset=['CHAIN', 'CO_OWNED']
).copy()

# Baseline model with heteroskedasticity-robust SE
baseline_robust = smf.ols(
    'Change_FTE ~ NJ',
    data=robust_sample
).fit(cov_type='HC3')

# Add restaurant-chain and company-ownership controls
controls_robust = smf.ols(
    'Change_FTE ~ NJ + C(CHAIN) + CO_OWNED',
    data=robust_sample
).fit(cov_type='HC3')

results_robustness = pd.DataFrame({
    'NJ estimate': [
        baseline_robust.params['NJ'],
        controls_robust.params['NJ']
    ],
    'Robust SE': [
        baseline_robust.bse['NJ'],
        controls_robust.bse['NJ']
    ],
    'p-value': [
        baseline_robust.pvalues['NJ'],
        controls_robust.pvalues['NJ']
    ],
    'CI low': [
        baseline_robust.conf_int().loc['NJ', 0],
        controls_robust.conf_int().loc['NJ', 0]
    ],
    'CI high': [
        baseline_robust.conf_int().loc['NJ', 1],
        controls_robust.conf_int().loc['NJ', 1]
    ],
    'N': [
        int(baseline_robust.nobs),
        int(controls_robust.nobs)
    ]
}, index=['Baseline', 'Chain + ownership controls'])

print(results_robustness.round(4))
print(controls_robust.summary().tables[1])

                            NJ estimate  Robust SE  p-value  CI low  CI high  \
Baseline                         2.7500     1.3505   0.0417  0.1031   5.3969   
Chain + ownership controls       2.7846     1.3578   0.0403  0.1234   5.4458   

                              N  
Baseline                    384  
Chain + ownership controls  384  
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -1.8277      1.555     -1.176      0.240      -4.875       1.220
C(CHAIN)[T.2]     0.2408      1.055      0.228      0.819      -1.827       2.308
C(CHAIN)[T.3]    -2.5884      1.151     -2.248      0.025      -4.845      -0.332
C(CHAIN)[T.4]    -0.1910      1.728     -0.111      0.912      -3.577       3.195
NJ                2.7846      1.358      2.051      0.040       0.123       5.446
CO_OWNED          0.3625      0.906      0.400      0.689      -1.413       2.138
